# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdulah-naeem/FlyRank-ml-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

This is a **Scoring** task. For deciding which pages an editor should fix first, we need to rank content based on a priority score (opportunity for refresh or risk of decline). We are producing a ranked action queue, meaning a continuous score representing opportunity or risk is more useful than a binary classification, as it allows us to rank pages from highest priority to lowest.

In [7]:
# Code backing: Sketching what the model output (a score) would look like
import numpy as np
import pandas as pd

# A scoring model outputs a continuous probability or risk score rather than just a 1/0 label.
mock_scores = np.random.uniform(0, 1, size=5)
mock_output = pd.DataFrame({
    'content_id': ['page_A', 'page_B', 'page_C', 'page_D', 'page_E'],
    'decline_risk_score': mock_scores
}).sort_values(by='decline_risk_score', ascending=False)

display(mock_output)


,content_id,decline_risk_score
1,page_B,0.999361
0,page_A,0.439268
2,page_C,0.416668
4,page_E,0.367721
3,page_D,0.047622


## 2. Target or proxy

The ideal target is **future 30-day traffic decline**, an observed outcome measured in a later time window (e.g., using the warehouse dataset). Using an observed future outcome is much better than a proxy rule from the starter dataset (like `is_declining_label`, which is just a defined rule based on `trend_pct`). Predicting a defined rule just teaches the model to learn the formula, whereas predicting future observed traffic teaches the model about the real world.

In [8]:
# Code backing: Sketching the target column using a derived proxy label as an example
import pandas as pd

# Load data to show the proxy
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

# We want an observed binary target (1 = declined, 0 = stable/grown).
# For now, we sketch this by creating a proxy from trend_direction.
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)
target_sketch = df[['client_id', 'is_declining_label']].head(5)
display(target_sketch)
print(f"Target distribution:\n{df['is_declining_label'].value_counts(normalize=True)}")


,client_id,is_declining_label
0,client_f369cb89fc,1
1,client_4e07408562,1
2,client_7f2253d7e2,1
3,client_19581e27de,0
4,client_3fdba35f04,1


Target distribution:
is_declining_label
1    0.542067
0    0.457933
Name: proportion, dtype: float64


## 3. Success metric

We will use multiple metrics to defend success:
1. **Precision@K** (e.g., Precision@20): Of the top 20 pages we score highest for a refresh, how many actually experienced a measurable decline or had a real opportunity? This directly measures the quality of the ranked queue an editor would work from.
2. **ROC-AUC**: To measure the model's overall ability to distinguish between pages that will decline and those that won't across all thresholds.

In [9]:
# Code backing: Demonstrating how Precision@K would be calculated
from sklearn.metrics import precision_score

# Let's say we have 10 pages, 3 of which actually declined (True = 1)
y_true = [1, 0, 1, 0, 0, 1, 0, 0, 0, 0]

# Our model scores them, and we pick the Top K=3 highest scored pages
# Suppose 2 out of our top 3 picks actually declined
y_pred_top_3 = [1, 1, 1, 0, 0, 0, 0, 0, 0, 0]

p_at_3 = precision_score(y_true, y_pred_top_3)
print(f"Precision@3 for this mock queue is: {p_at_3:.2f} (We want this as close to 1.0 as possible)")


Precision@3 for this mock queue is: 0.67 (We want this as close to 1.0 as possible)


## 4. The unit of analysis, as a real dataframe

One row = one pseudonymized content item (e.g., a single article or page) for a specific client.

In [10]:
import pandas as pd

# Load the starter dataset for Lane 2
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

# Show the unit of analysis (one row = one content item)
display(df.head(3))
print(f"Dataset shape: {df.shape}")


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9


Dataset shape: (30000, 44)


## 5. Why ML beats a fixed rule here

A fixed rule (like `if traffic_dropped > 10% then refresh`) is too brittle. ML beats a fixed rule here because we are dealing with multiple tangled signals (e.g., engagement rates, missing keyword data, scroll depths, and various client contexts). An if-statement cannot easily weigh 44 different dimensions or adjust for missingness patterns across 32 different clients, whereas an ML model can find the nuanced interactions between these messy signals to score opportunities more accurately.

In [11]:
# Code backing: Demonstrating the brittleness of a fixed rule
import pandas as pd
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

# A naive fixed rule: "Refresh if traffic is dropping by more than 10% and CTR is less than 0.5%"
# Let's see how many items that rule actually captures:
naive_rule_matches = df[(df['trend_pct'] < -10) & (df['ctr'] < 0.5)]
print(f"A naive rule only flags {len(naive_rule_matches)} out of {len(df)} pages.")

# But we have 44 features we could be using to find nuanced decay patterns!
print(f"Total features available for ML to find complex patterns: {df.shape[1]}")


A naive rule only flags 15786 out of 30000 pages.
Total features available for ML to find complex patterns: 44


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.